# Import dan Konstanta

In [5]:
import os
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

RANDOM_SEED = 42
TEST_SIZE = 0.2
N_SPLITS = 5

FEATURED_PATH = r"D:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\data\processed\application_train_featured.csv"
SPLIT_PATH = r"D:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\data\processed\holdout_split.json"
FOLDS_PATH = r"D:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\data\processed\cv_folds.json"

#  Definisikan Tiga Fungsi

In [6]:
def load_featured_dataset(path=FEATURED_PATH):
    df = pd.read_csv(path)
    print(f"Shape loaded: {df.shape}")
    return df

def create_or_load_holdout_split(df, id_col="SK_ID_CURR", target_col="TARGET",
                                  test_size=TEST_SIZE, seed=RANDOM_SEED, path=SPLIT_PATH):
    if os.path.exists(path):
        with open(path, "r") as f:
            split_info = json.load(f)
        print("Holdout split dimuat dari file existing.")
        return split_info
    dev_idx, test_idx = train_test_split(
        df.index, test_size=test_size, stratify=df[target_col], random_state=seed
    )
    split_info = {
        "random_seed": seed,
        "test_size": test_size,
        "development_ids": df.loc[dev_idx, id_col].tolist(),
        "test_ids": df.loc[test_idx, id_col].tolist(),
    }
    with open(path, "w") as f:
        json.dump(split_info, f)
    print("Holdout split baru dibuat dan disimpan.")
    return split_info

def create_or_load_cv_folds(df, split_info, id_col="SK_ID_CURR", target_col="TARGET",
                             n_splits=N_SPLITS, seed=RANDOM_SEED, path=FOLDS_PATH):
    if os.path.exists(path):
        with open(path, "r") as f:
            fold_info = json.load(f)
        print("CV fold dimuat dari file existing.")
        return fold_info
    dev_ids = split_info["development_ids"]
    dev_df = df[df[id_col].isin(dev_ids)].reset_index(drop=True)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_assignment = np.full(len(dev_df), -1)
    for fold_idx, (_, val_idx) in enumerate(skf.split(dev_df, dev_df[target_col])):
        fold_assignment[val_idx] = fold_idx
    fold_info = {
        "random_seed": seed,
        "n_splits": n_splits,
        "sk_id_curr": dev_df[id_col].tolist(),
        "fold_assignment": fold_assignment.tolist(),
    }
    with open(path, "w") as f:
        json.dump(fold_info, f)
    print("CV fold baru dibuat dan disimpan.")
    return fold_info

# Jalankan dan Verifikasi

In [7]:
df = load_featured_dataset()
assert df.shape == (307511, 73), f"Shape tidak sesuai! Didapat: {df.shape}"

split_info = create_or_load_holdout_split(df)
dev_df = df[df["SK_ID_CURR"].isin(split_info["development_ids"])]
test_df = df[df["SK_ID_CURR"].isin(split_info["test_ids"])]

print("Development set:", dev_df.shape)
print(dev_df["TARGET"].value_counts(normalize=True).round(4))
print("\nTest set:", test_df.shape)
print(test_df["TARGET"].value_counts(normalize=True).round(4))

fold_info = create_or_load_cv_folds(df, split_info)
fold_arr = np.array(fold_info["fold_assignment"])
sk_ids_arr = np.array(fold_info["sk_id_curr"])

print("\nDistribusi 5-Fold CV:")
for f in range(N_SPLITS):
    fold_ids = sk_ids_arr[fold_arr == f]
    fold_target = df[df["SK_ID_CURR"].isin(fold_ids)]["TARGET"]
    print(f"Fold {f}: n={len(fold_ids)}, proporsi TARGET=1 = {fold_target.mean():.4f}")

Shape loaded: (307511, 73)
Holdout split dimuat dari file existing.
Development set: (246008, 73)
TARGET
0    0.9193
1    0.0807
Name: proportion, dtype: float64

Test set: (61503, 73)
TARGET
0    0.9193
1    0.0807
Name: proportion, dtype: float64
CV fold dimuat dari file existing.


KeyError: 'sk_id_curr'

# RareCategoryGrouper

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=50, other_label="Other"):
        self.threshold = threshold
        self.other_label = other_label

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.frequent_categories_ = {}
        for col in X.columns:
            counts = X[col].value_counts()
            self.frequent_categories_[col] = counts[counts >= self.threshold].index.tolist()
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for col in X.columns:
            allowed = self.frequent_categories_.get(col, [])
            X[col] = X[col].where(X[col].isin(allowed), self.other_label)
        return X

# Dua Fungsi Preprocessing

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CATEGORICAL_COLUMNS = [
    "OCCUPATION_TYPE", "ORGANIZATION_TYPE", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE",
    "CODE_GENDER", "WALLSMATERIAL_MODE", "EMERGENCYSTATE_MODE", "HOUSETYPE_MODE",
    "NAME_FAMILY_STATUS", "NAME_HOUSING_TYPE", "NAME_CONTRACT_TYPE", "FLAG_OWN_CAR",
    "NAME_TYPE_SUITE", "WEEKDAY_APPR_PROCESS_START", "FLAG_OWN_REALTY",
]

def build_linear_mlp_preprocessor(feature_list, rare_threshold=50):
    categorical_cols = [c for c in feature_list if c in CATEGORICAL_COLUMNS]
    numeric_cols = [c for c in feature_list if c not in CATEGORICAL_COLUMNS]

    categorical_pipeline = Pipeline([
        ("rare_grouper", RareCategoryGrouper(threshold=rare_threshold)),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer([
        ("cat", categorical_pipeline, categorical_cols),
        ("num", StandardScaler(), numeric_cols),
    ])
    return preprocessor

def build_tree_preprocessor(feature_list, rare_threshold=50):
    categorical_cols = [c for c in feature_list if c in CATEGORICAL_COLUMNS]
    numeric_cols = [c for c in feature_list if c not in CATEGORICAL_COLUMNS]

    categorical_pipeline = Pipeline([
        ("rare_grouper", RareCategoryGrouper(threshold=rare_threshold)),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer([
        ("cat", categorical_pipeline, categorical_cols),
        ("num", "passthrough", numeric_cols),
    ])
    return preprocessor

# Verifikasi pada Fold Pertama Saja

In [ ]:
FEATURE_METADATA_PATH = r"D:\Kerjaan\Home Creadit\Project-1-Home-Credit-Default-Risk-main\data\processed\feature_engineering_metadata.json"

with open(FEATURE_METADATA_PATH, "r") as f:
    feature_metadata = json.load(f)

linear_mlp_features = feature_metadata["linear_mlp_features"]

fold0_train_ids = sk_ids_arr[fold_arr != 0]
X_fold0_train = df[df["SK_ID_CURR"].isin(fold0_train_ids)][linear_mlp_features]

preprocessor = build_linear_mlp_preprocessor(linear_mlp_features)
X_transformed = preprocessor.fit_transform(X_fold0_train)

print("n_features sebelum encoding:", len(linear_mlp_features))
print("n_features setelah encoding:", X_transformed.shape[1])

n_features sebelum encoding: 66
n_features setelah encoding: 183


#  cek cardinality per kolom kategorikal setelah rare grouping 

In [ ]:
rare_grouper = preprocessor.named_transformers_["cat"].named_steps["rare_grouper"]
for col, categories in rare_grouper.frequent_categories_.items():
    print(f"{col}: {len(categories)} kategori dipertahankan (+ 'Other')")

OCCUPATION_TYPE: 19 kategori dipertahankan (+ 'Other')
ORGANIZATION_TYPE: 55 kategori dipertahankan (+ 'Other')
NAME_INCOME_TYPE: 4 kategori dipertahankan (+ 'Other')
NAME_EDUCATION_TYPE: 5 kategori dipertahankan (+ 'Other')
CODE_GENDER: 2 kategori dipertahankan (+ 'Other')
WALLSMATERIAL_MODE: 8 kategori dipertahankan (+ 'Other')
EMERGENCYSTATE_MODE: 3 kategori dipertahankan (+ 'Other')
HOUSETYPE_MODE: 4 kategori dipertahankan (+ 'Other')
NAME_FAMILY_STATUS: 5 kategori dipertahankan (+ 'Other')
NAME_HOUSING_TYPE: 6 kategori dipertahankan (+ 'Other')
NAME_CONTRACT_TYPE: 2 kategori dipertahankan (+ 'Other')
FLAG_OWN_CAR: 2 kategori dipertahankan (+ 'Other')
NAME_TYPE_SUITE: 7 kategori dipertahankan (+ 'Other')
WEEKDAY_APPR_PROCESS_START: 7 kategori dipertahankan (+ 'Other')
FLAG_OWN_REALTY: 2 kategori dipertahankan (+ 'Other')


#  Verifikasi build_tree_preprocessor

In [ ]:
tree_features = feature_metadata["tree_features"]

X_fold0_train_tree = df[df["SK_ID_CURR"].isin(fold0_train_ids)][tree_features]

tree_preprocessor = build_tree_preprocessor(tree_features)
X_transformed_tree = tree_preprocessor.fit_transform(X_fold0_train_tree)

print("n_features sebelum encoding (tree):", len(tree_features))
print("n_features setelah encoding (tree):", X_transformed_tree.shape[1])

n_features sebelum encoding (tree): 66
n_features setelah encoding (tree): 183
